<a href="https://colab.research.google.com/github/mehmetgul/NLP/blob/main/CS_7347_Natural_Language_Processing_Homework_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import re
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
import math

print("Libraries loaded successfully.")

Libraries loaded successfully.


Q1 - Please write regular expressions for the following (20 points)
a.

1.   All binary strings. Examples: 1001, 1011, 1111, etc.
2.   The email address contains only letters, and @, \. Symbols (both lower and upper cases). Examples: alice@gmail.com, bob@yahoo.com, etc.
3.   Valid integer numbers. Examples: 1, 12843, -89232, +1262, etc.
4.    Valid phone number that contains ten (10) digits. Consider the phone number formats below:
      *   xxx-xxx-xxxx
      *   (xxx) xxx-xxxx
      *   Examples: 453-126-4570, (453) 126-4560


In [6]:
# --- 1. Binary Strings ---
# Pattern: Start with ^, contain only 0 or 1, end with $
binary_pattern = r'^[01]+$'
binary_tests = ['1001', '1011', '1111', '1021', 'abc']

print("--- Binary String Tests ---")
for t in binary_tests:
    print(f"'{t}': {bool(re.match(binary_pattern, t))}")

# --- 2. Email Addresses ---
# Pattern: Letters -> @ -> Letters -> . -> Letters
email_pattern = r'^[a-zA-Z]+@[a-zA-Z]+\.[a-zA-Z]+$'
email_tests = ['alice@gmail.com', 'bob@yahoo.com', 'alice123@gmail.com', 'alice@gmail']

print("\n--- Email Tests ---")
for t in email_tests:
    print(f"'{t}': {bool(re.match(email_pattern, t))}")

# --- 3. Integer Numbers ---
# Pattern: Optional sign (+ or -), followed by digits
int_pattern = r'^[+-]?[0-9]+$'
int_tests = ['1', '12843', '-89232', '+1262', '12.5', '--5']

print("\n--- Integer Tests ---")
for t in int_tests:
    print(f"'{t}': {bool(re.match(int_pattern, t))}")

# --- 4. Phone Numbers ---
# Pattern: xxx-xxx-xxxx OR (xxx) xxx-xxxx
phone_pattern = r'^(\d{3}-\d{3}-\d{4}|\(\d{3}\) \d{3}-\d{4})$'
phone_tests = ['453-126-4570', '(453) 126-4560', '1234567890', '12-12-12']

print("\n--- Phone Number Tests ---")
for t in phone_tests:
    print(f"'{t}': {bool(re.match(phone_pattern, t))}")

--- Binary String Tests ---
'1001': True
'1011': True
'1111': True
'1021': False
'abc': False

--- Email Tests ---
'alice@gmail.com': True
'bob@yahoo.com': True
'alice123@gmail.com': False
'alice@gmail': False

--- Integer Tests ---
'1': True
'12843': True
'-89232': True
'+1262': True
'12.5': False
'--5': False

--- Phone Number Tests ---
'453-126-4570': True
'(453) 126-4560': True
'1234567890': False
'12-12-12': False


Q2 - Determine the number of tokens and vocabulary, and types from the text below. Please list them in your answer too. **(5 points)**

**Text:** “The quick brown fox jumps over the lazy dog.”

In [7]:
# --- Q2: Tokens and Vocabulary ---
# We treat punctuation as a token for this academic exercise
# Simple tokenization by splitting on whitespace and separating the period
text_raw = "The quick brown fox jumps over the lazy dog."
tokens = text_raw.replace('.', ' .').split()

# Vocabulary (Types) - using a set to find unique words (case-insensitive)
# We lower() them to ensure "The" and "the" count as the same type
vocabulary = set([t.lower() for t in tokens])

print(f"Original Text: {text_raw}\n")
print(f"Tokens (List): {tokens}")
print(f"Number of Tokens: {len(tokens)}")
print(f"Vocabulary (Types): {vocabulary}")
print(f"Number of Types: {len(vocabulary)}")

Original Text: The quick brown fox jumps over the lazy dog.

Tokens (List): ['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '.']
Number of Tokens: 10
Vocabulary (Types): {'lazy', 'the', 'dog', '.', 'fox', 'over', 'jumps', 'brown', 'quick'}
Number of Types: 9


Q3 - Write down all the steps of text normalization and give an example for each step. **(5 points)**

In [8]:
# --- Q3: Normalization Steps Example ---
print("\n--- Text Normalization Steps ---")

text_raw = "The quick brown fox jumps over the lazy dog."

def normalize_text_step_by_step(text):
    # Step 1: Segmentation
    print(f"1. Segmentation (Raw): {text}")

    # Step 2: Tokenization
    tokens = text.split()
    print(f"2. Tokenization: {tokens}")

    # Step 3: Case Folding (Lowercasing)
    tokens_lower = [t.lower() for t in tokens]
    print(f"3. Case Folding: {tokens_lower}")

    # Step 4: Noise Removal (Removing the period)
    tokens_clean = [t.replace('.', '') for t in tokens_lower]
    print(f"4. Noise Removal (Punctuation): {tokens_clean}")

    return tokens_clean

final_tokens = normalize_text_step_by_step(text_raw)


--- Text Normalization Steps ---
1. Segmentation (Raw): The quick brown fox jumps over the lazy dog.
2. Tokenization: ['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog.']
3. Case Folding: ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog.']
4. Noise Removal (Punctuation): ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']


Q4 - We know how to compute similarity distance between two given strings using the edit distance algorithm. (25 points)
1. Please write down the distance matrix for the following strings. Consider space “ “ as a single character. (15 points)
    - Strings 1: Spokesman confirms
    - String 2: Spokeswoman said
2. List down all the operations you need to perform. Please show backtracing matrix to validate your answer for the above example strings. (10 points)

In [9]:
def min_edit_distance(source, target):
    # Costs
    DEL_COST = 1
    INS_COST = 1
    SUB_COST = 2

    n = len(source)
    m = len(target)

    # Initialize Matrix
    # We add 1 to dimensions for the empty string "" case
    D = np.zeros((n+1, m+1), dtype=int)

    # Initialization (0th row and 0th column)
    for i in range(1, n+1):
        D[i, 0] = D[i-1, 0] + DEL_COST
    for j in range(1, m+1):
        D[0, j] = D[0, j-1] + INS_COST

    # Fill Matrix
    for i in range(1, n+1):
        for j in range(1, m+1):
            # Check for substitution cost (0 if match, 2 if mismatch)
            cost = 0 if source[i-1] == target[j-1] else SUB_COST

            D[i, j] = min(
                D[i-1, j] + DEL_COST,
                D[i, j-1] + INS_COST,
                D[i-1, j-1] + cost # Substitution
            )

    # Pandas DataFrame for visualization
    # We add '#' to represent the empty string start
    cols = ['#'] + list(target)
    rows = ['#'] + list(source)
    df = pd.DataFrame(D, index=rows, columns=cols)

    return df, D[n, m]

# Strings from homework
str1 = "Spokesman confirms"
str2 = "Spokeswoman said"

matrix_df, distance = min_edit_distance(str1, str2)

print(f"Edit Distance: {distance}")
print("\n--- Distance Matrix ---")
display(matrix_df)

Edit Distance: 12

--- Distance Matrix ---


,#,S,p,o,k,e,s,w,o,m,a,n,,s,a,i,d
#,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
S,1,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
p,2,1,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
o,3,2,1,0,1,2,3,4,5,6,7,8,9,10,11,12,13
k,4,3,2,1,0,1,2,3,4,5,6,7,8,9,10,11,12
e,5,4,3,2,1,0,1,2,3,4,5,6,7,8,9,10,11
s,6,5,4,3,2,1,0,1,2,3,4,5,6,7,8,9,10
m,7,6,5,4,3,2,1,2,3,2,3,4,5,6,7,8,9
a,8,7,6,5,4,3,2,3,4,3,2,3,4,5,6,7,8
n,9,8,7,6,5,4,3,4,5,4,3,2,3,4,5,6,7


Q5 - Please formulate your language model for the following text. Show the details of your LM formulation. (25 points)
**Text:** “The day was grey and bitter cold, and the dogs would not take the scent. The big black bitch had taken one sniff at the bear tracks, backed off, and skulked back to the pack with her tail between her legs.”
1. Unigram model (10 points)
2. Bigram model (15 points)



**A. Unigram Model**

The likelihood of a word $w_i$ is the number of occurrences of a given word divided by the number of words in total $N$.

$$P(w_i) = \frac{Count(w_i)}{N}$$

*Where $N = 41$ (Total tokens).*

**Examples:**
* $$P(\text{the}) = \frac{6}{41} \approx 0.1463$$
* $$P(\text{and}) = \frac{3}{41} \approx 0.0732$$
* $$P(\text{her}) = \frac{2}{41} \approx 0.0488$$

---

**B. Bigram Model**
The likelihood of a word $w_i$ depends on the previous word $w_{i-1}$.

$$P(w_i | w_{i-1}) = \frac{Count(w_{i-1}, w_i)}{Count(w_{i-1})}$$

**Examples:**
* $$P(\text{day} | \text{the}) = \frac{Count(\text{the day})}{Count(\text{the})} = \frac{1}{6} \approx 0.1667$$
* $$P(\text{the} | \text{and}) = \frac{Count(\text{and the})}{Count(\text{and})} = \frac{1}{3} \approx 0.3333$$
* $$P(\text{off} | \text{backed}) = \frac{Count(\text{backed off})}{Count(\text{backed})} = \frac{1}{1} = 1.0$$

In [10]:
lm_text = """The day was grey and bitter cold, and the dogs would not take the scent.
The big black bitch had taken one sniff at the bear tracks, backed off,
and skulked back to the pack with her tail between her legs."""

# Normalize
# Remove punctuation, lowercase, split
clean_lm_text = re.sub(r'[^\w\s]', '', lm_text).lower().split()

print(f"Total Words (N): {len(clean_lm_text)}")

# --- Unigram Model ---
unigram_counts = Counter(clean_lm_text)
total_count = sum(unigram_counts.values())

print("\n--- Unigram Probabilities (Top 5) ---")
# P(w) = count(w) / N
for word, count in unigram_counts.most_common(5):
    print(f"P({word}) = {count}/{total_count} = {count/total_count:.4f}")

# --- Bigram Model ---
# Create bigrams
bigrams = list(zip(clean_lm_text, clean_lm_text[1:]))
bigram_counts = Counter(bigrams)

print("\n--- Bigram Probabilities (Sample) ---")
# P(w_i | w_i-1) = count(w_i-1, w_i) / count(w_i-1)
samples = [('the', 'day'), ('and', 'the'), ('backed', 'off')]

for w1, w2 in samples:
    bg_count = bigram_counts[(w1, w2)]
    u_count = unigram_counts[w1]
    prob = bg_count / u_count if u_count > 0 else 0
    print(f"P({w2} | {w1}) = {bg_count}/{u_count} = {prob:.2f}")

Total Words (N): 41

--- Unigram Probabilities (Top 5) ---
P(the) = 6/41 = 0.1463
P(and) = 3/41 = 0.0732
P(her) = 2/41 = 0.0488
P(day) = 1/41 = 0.0244
P(was) = 1/41 = 0.0244

--- Bigram Probabilities (Sample) ---
P(day | the) = 1/6 = 0.17
P(the | and) = 1/3 = 0.33
P(off | backed) = 1/1 = 1.00


Q6 - You are given a training set of 30 numbers that consists of 21 zeros and 1
each of the other digits 1-9. Now we see the following test set: 0 0 0 0 0 3 0 0 0 0. What is the unigram perplexity? (20 points)

In [11]:
# --- Training Data Setup ---
# 21 zeros, and one of each digit 1-9
count_0 = 21
count_others = 1 # for 1, 2, 3... 9
total_training_tokens = 30

# Calculate Probabilities from Training Data
p_zero = count_0 / total_training_tokens
p_other = count_others / total_training_tokens # P(1), P(2)... P(9) are all equal

print(f"P(0): {p_zero}")
print(f"P(3): {p_other}") # We need P(3) for the test set

# --- Test Data ---
# Sequence: 0 0 0 0 0 3 0 0 0 0
# 9 zeros, 1 three
test_sequence_length = 10
zeros_in_test = 9
threes_in_test = 1

# Calculate Probability for Test Sequence P(S)
# P(S) = P(0)^9 * P(3)^1
p_sequence = (p_zero ** zeros_in_test) * (p_other ** threes_in_test)

print(f"Probability of Sequence: {p_sequence}")

# --- Perplexity Calculation ---
# PP(S) = P(S) ^ (-1/N)
perplexity = p_sequence ** (-1 / test_sequence_length)

print(f"Perplexity: {perplexity}")

P(0): 0.7
P(3): 0.03333333333333333
Probability of Sequence: 0.0013451202333333327
Perplexity: 1.936974438134067
